In [1]:
# === Cell 1: Imports, precision & banner ===

"""
Fine-tune Arctic-Text2SQL model with LoRA adapters.
This version loads the Spider dataset directly from Hugging Face (xlangai/spider)
instead of a local JSON file.
"""

import os
import json
from datetime import datetime

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

# Slight speed boost on modern GPUs
torch.set_float32_matmul_precision("high")

print("="*80)
print("FINE-TUNING ARCTIC-TEXT2SQL MODEL")
print("="*80)
print("This will add LoRA adapters to improve the model while preserving base performance.")
print()


d:\collegestuff\DATA_266\Text-To-SQL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FINE-TUNING ARCTIC-TEXT2SQL MODEL
This will add LoRA adapters to improve the model while preserving base performance.



In [2]:
# === Cell 2: Configuration & CUDA check ===

# =========================
# Configuration
# =========================
BASE_MODEL = "Snowflake/Arctic-Text2SQL-R1-7B"  # Base model for fine-tuning
OUTPUT_DIR = "arctic_lora_model"                # Your fine-tuned adapters

# Hugging Face dataset settings (Spider)
HF_DATASET_NAME = "xlangai/spider"              # Hugging Face dataset id
HF_SPLIT = "train"                              # use Spider train split
TRAINING_SUBSET_SIZE = None                      # set to None to use full train (~7k)

# =========================
# Check CUDA
# =========================
if not torch.cuda.is_available():
    print("❌ CUDA not available. Fine-tuning requires GPU.")
    raise SystemExit(1)

device_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"✅ CUDA available: {device_name}")
print(f"✅ CUDA memory: {total_mem_gb:.1f} GB")
print()


✅ CUDA available: NVIDIA GeForce RTX 5070 Ti
✅ CUDA memory: 17.1 GB



In [3]:
# === Cell 3: Load base model & tokenizer (4-bit) ===

print("[1/7] Loading base model: Arctic-Text2SQL-R1-7B")
print("This may take a few minutes (downloading if first time)...")

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Base model loaded!")


[1/7] Loading base model: Arctic-Text2SQL-R1-7B
This may take a few minutes (downloading if first time)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.74s/it]


✅ Base model loaded!


In [4]:
# === Cell 4: Prepare model & add LoRA adapters ===

print("\n[2/7] Preparing model for training...")
model = prepare_model_for_kbit_training(model)

# Disable gradient checkpointing for speed (4-bit + LoRA on 16GB should be fine)
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
if hasattr(model, "config"):
    model.config.use_cache = False  # keep off during training

print("\n[3/7] Adding LoRA adapters...")
lora_config = LoraConfig(
    r=16,  # Rank (lower = smaller adapters, less overfitting)
    lora_alpha=32,  # Scaling factor
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(
    f"✅ Trainable params: {trainable_params:,} || "
    f"all params: {all_params:,} || "
    f"trainable%: {100 * trainable_params / all_params:.2f}%"
)



[2/7] Preparing model for training...

[3/7] Adding LoRA adapters...
✅ Trainable params: 40,370,176 || all params: 4,393,342,464 || trainable%: 0.92%


In [5]:
# === Cell 5: Load Spider dataset from Hugging Face ===

print("\n[4/7] Loading Spider dataset from Hugging Face...")
print(f"   Dataset: {HF_DATASET_NAME}, split='{HF_SPLIT}'")

dataset = load_dataset(HF_DATASET_NAME, split=HF_SPLIT)
print(f"✅ Loaded {len(dataset)} examples from Hugging Face")

# Use subset for quick fine-tuning (if specified)
if TRAINING_SUBSET_SIZE and TRAINING_SUBSET_SIZE < len(dataset):
    print(f"📊 Using subset of {TRAINING_SUBSET_SIZE} examples for quick fine-tuning")
    dataset = dataset.select(range(TRAINING_SUBSET_SIZE))

print(f"✅ Using {len(dataset)} examples for training")



[4/7] Loading Spider dataset from Hugging Face...
   Dataset: xlangai/spider, split='train'
✅ Loaded 7000 examples from Hugging Face
✅ Using 7000 examples for training


In [6]:
# === Cell 6: Format dataset into Alpaca-style prompts ===

print("\n[5/7] Formatting dataset into Alpaca-style prompts...")

INSTRUCTION_TEMPLATE = (
    "You are a powerful text-to-SQL model. Your job is to generate "
    "valid SQL queries for the given database and question."
)

def format_instruction(example):
    """
    Spider fields (from xlangai/spider):
      - db_id: database id
      - question: natural language question
      - query: target SQL
    """
    db_id = example["db_id"]
    question = example["question"]
    sql = example["query"]

    instruction = INSTRUCTION_TEMPLATE

    # You can enrich this later with schema text if you want.
    # For now we give db_id + question.
    input_text = f"DB_ID: {db_id}\nQ: {question}\nSQL:"

    text = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{sql}"""

    return {"text": text}

dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)
print("✅ Dataset formatted! (field used: 'text')")



[5/7] Formatting dataset into Alpaca-style prompts...
✅ Dataset formatted! (field used: 'text')


In [7]:
# === Cell 7: TrainingArguments ===

print("\n[6/7] Setting up training...")

use_bf16 = torch.cuda.is_bf16_supported()
print(f"✅ bfloat16 support: {use_bf16}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                 # Start with 1 epoch (can increase)
    per_device_train_batch_size=4,      # Slightly larger batch
    gradient_accumulation_steps=2,      # Effective batch size = 8
    warmup_steps=50,
    learning_rate=5e-5,                 # Lower LR to preserve base performance
    fp16=False,
    bf16=use_bf16,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="no",
    optim="adamw_torch",                # Safe default on Windows
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
    gradient_checkpointing=False,       # Explicitly off for speed
    dataloader_num_workers=0,           # Windows compatibility
    max_grad_norm=1.0,
)



[6/7] Setting up training...
✅ bfloat16 support: True


In [8]:
import trl
trl.__version__

'0.25.1'

In [9]:
# === Cell 8: Create SFTTrainer (TRL new API) ===

print("\n[7/7] Creating trainer...")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    # TRL >= 0.8 uses `processing_class` instead of `tokenizer`
    processing_class=tokenizer,
)

print("\n" + "="*80)
print("TRAINING CONFIGURATION")
print("="*80)
print(f"Base Model: {BASE_MODEL}")
print(f"Dataset size (after subset): {len(dataset)} examples")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(
    f"Effective batch size: "
    f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}"
)
print(f"Learning rate: {training_args.learning_rate}")
print(f"Tokenizer max length: {tokenizer.model_max_length}")
print("="*80)
print()



[7/7] Creating trainer...

TRAINING CONFIGURATION
Base Model: Snowflake/Arctic-Text2SQL-R1-7B
Dataset size (after subset): 7000 examples
Epochs: 3
Batch size: 4
Gradient accumulation: 2
Effective batch size: 8
Learning rate: 5e-05
Tokenizer max length: 131072



In [10]:
# === Cell 9: Start training ===

print("Starting training...")
print("="*80)

train_result = trainer.train()

print("\nTraining finished!")
print(train_result)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151645}.


Starting training...


Step,Training Loss
10,2.571900
20,2.384400
30,1.921600
40,1.016800
50,0.703700
60,0.619600
70,0.576100
80,0.578500
90,0.554800
100,0.557000



Training finished!
TrainOutput(global_step=2625, training_loss=0.312153647059486, metrics={'train_runtime': 12218.4563, 'train_samples_per_second': 1.719, 'train_steps_per_second': 0.215, 'total_flos': 1.3266631431911424e+17, 'train_loss': 0.312153647059486, 'entropy': 0.21613651812076567, 'num_tokens': 2510802.0, 'mean_token_accuracy': 0.9404707252979279, 'epoch': 3.0})


In [11]:
# === Cell 10: Save fine-tuned model & training summary ===

print("\n" + "="*80)
print("SAVING FINE-TUNED MODEL")
print("="*80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

summary = {
    "base_model": BASE_MODEL,
    "fine_tuning_method": "LoRA",
    "dataset_source": HF_DATASET_NAME,
    "dataset_split": HF_SPLIT,
    "dataset_size": len(dataset),
    "epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "training_date": datetime.now().isoformat(),
    "trainable_params": trainable_params,
    "total_params": all_params,
    "trainable_percentage": f"{100 * trainable_params / all_params:.2f}%"
}

with open(os.path.join(OUTPUT_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"✅ Fine-tuned model saved to: {OUTPUT_DIR}/")
print(f"✅ Training summary saved to: {OUTPUT_DIR}/training_summary.json")
print()
print("="*80)
print("TRAINING COMPLETE")
print("="*80)
print()
print("Next steps:")
print("1. Start your inference server (finetuned_arctic_server.py pointing to this OUTPUT_DIR).")
print("2. Point your Streamlit app to the server's URL.")
print("3. Compare performance with base Arctic and GPT-4o-mini.")
print("="*80)



SAVING FINE-TUNED MODEL
✅ Fine-tuned model saved to: arctic_lora_model/
✅ Training summary saved to: arctic_lora_model/training_summary.json

TRAINING COMPLETE

Next steps:
1. Start your inference server (finetuned_arctic_server.py pointing to this OUTPUT_DIR).
2. Point your Streamlit app to the server's URL.
3. Compare performance with base Arctic and GPT-4o-mini.
